In [1]:
#import dataiku

import pandas as pd, numpy as np

#from dataiku import pandasutils as pdu



In [2]:
# -------------------------------------------------------------------------------- NOTEBOOK-CELL: CODE

import statsmodels

import statsmodels.api as sm

import statsmodels.formula.api as smf



In [4]:
# -------------------------------------------------------------------------------- NOTEBOOK-CELL: CODE

import os

import sys

from scipy import stats

import matplotlib.pyplot as plt

import seaborn as sns

# %matplotlib inline

import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.linear_model import LinearRegression  # Add this import

# -------------------------------------------------------------------------------- NOTEBOOK-CELL: CODE



In [5]:
from datetime import date

import holidays

from workalendar.europe import France

import datetime


In [ ]:
data = pd.read_csv(r"C:\Users\utilisateur\Downloads\wetransfer_histo_gazpar_merged-3-csv-gz_2026-01-30_1023\histo_gazpar_merged(1).csv.gz")


In [9]:
data.shape

(2797952, 35)

In [10]:
data['pce']=data['pce'].astype(str)


In [11]:

def add_leading_zeros(pce_value):

    if len(pce_value) <= 13:

        # Pad the string with leading zeros to make it 14 characters long

        return pce_value.zfill(14)

    else:

        return pce_value

# Apply the function to the 'pce' column

data['pce'] = data['pce'].apply(add_leading_zeros)


In [12]:
data = data.sort_values(by=['pce', 'gasday'])

data['trend'] = data.groupby('pce').cumcount() + 1

In [13]:
data['gasday'] = pd.to_datetime(data['gasday']).dt.date

# Function to determine if a day is a workday (Mon-Fri) or weekend (Sat-Sun)

def is_workday(date):

   return 1 if date.weekday() < 5 else 0

# Apply the function to create the Workday column

data['workday'] = data['gasday'].apply(is_workday)

data['weekend'] = 1-data['workday']

In [14]:
# ## Add Holidays

# -------------------------------------------------------------------------------- NOTEBOOK-CELL: CODE

# Assuming col 'year'

unique_years = data['year'].unique()

# Convert unique_years to integers and create a list

unique_years_list = unique_years.astype(int).tolist()

# Create an empty dictionary to store the holidays for each year

fr_holidays = {}

# Iterate through unique years and get the public holidays for each year

for year in unique_years_list:

    fr_holidays[year] = holidays.France(years=[year])

# Flatten the holiday dictionary for easier lookup

all_holidays = {date: name for year in fr_holidays for date, name in fr_holidays[year].items()}

all_holidays


# Create a new column indicating whether each date is a public holiday (1 or 0)

data['Is_Public_Holiday'] = data['gasday'].apply(lambda x: 1 if x in all_holidays else 0)

# Create a column with the holiday name or an empty string if it is not a holiday

data['Holiday_Name'] = data['gasday'].apply(lambda x: all_holidays.get(x, ''))


In [ ]:
# Determine if it's a non-working day

data['non_working_day'] = data['weekend'] | data['Is_Public_Holiday']


In [17]:
# Count observations per category

data['count'] = data.groupby('pce')['pce'].transform('count')

In [18]:
# Check for missing values and drop rows with missing values

data_nona = data.dropna(subset=['valeur_energie_conso', 'hdd', 'trend', 'pce'])


In [20]:
# Subset the dataset where column 'is_outlier' is not equal to -1

# data_nona = data_nona[data_nona['is_outlier'] != -1]

# data_nona.shape

data_nona = data_nona[(data_nona['hdd'] != 0) & (data['valeur_energie_conso'] != 0)]

In [ ]:
# FE weights are related to overall residual variance (Captures "leftover" variability within groups after accounting for both fixed and random effects), RE varince (Captures how much group-specific deviations (random effects) vary around the population average (fixed effects).the RE variance-covariance matrix is used to weight the FE estimates and their uncertainty. The FE estimates are derived by "pooling" group-specific effects, with pooling weights determined by:

# The relative magnitude of within-group vs. between-group variance,

# Correlations between random effects (e.g., intercept-slope covariance).

# -------------------------------------------------------------------------------- NOTEBOOK-CELL: MARKDOWN

# ## Models

In [22]:
#%%time

lmm_default = smf.mixedlm(

    formula='valeur_energie_conso ~ hdd + trend + Is_Public_Holiday',

    data=data_nona,

    groups="pce",

    re_formula="~hdd + 1"

).fit()

lmm_default.summary()

# print(lmm_default.cov_re)




c:\Users\utilisateur\Desktop\programmation\Statap\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\utilisateur\Desktop\programmation\Statap\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
c:\Users\utilisateur\Desktop\programmation\Statap\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\utilisateur\Desktop\programmation\Statap\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with cg
  warnings.warn(
c:\Users\utilisateur\Desktop\programmation\Statap\.venv\Lib\site-packages\statsmodels

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:            MixedLM Dependent Variable: valeur_energie_conso
No. Observations: 1481335 Method:             REML                
No. Groups:       7957    Scale:              1622429.3752        
Min. group size:  1       Log-Likelihood:     -12694363.1637      
Max. group size:  429     Converged:          No                  
Mean group size:  186.2                                           
------------------------------------------------------------------
                      Coef.   Std.Err.   z    P>|z|  [0.025 0.975]
------------------------------------------------------------------
Intercept               8.565    2.770  3.093 0.002   3.137 13.994
hdd                     5.823    0.249 23.412 0.000   5.336  6.311
trend                   0.006    0.008  0.705 0.481  -0.010  0.021
Is_Public_Holiday      -3.646    6.033 -0.604 0.546 -15.470  8.178
pce Var              8552.859                                     
pce x hdd Cov        -556.955                                     
hdd Var                37.010                                     
==================================================================

"""

In [23]:
df_random_effects_default = pd.DataFrame.from_dict(lmm_default.random_effects, orient='index')


In [24]:
# Add fixed effects to the DataFrame

df_random_effects_default['Fixed intercept'] = lmm_default.fe_params.loc['Intercept']

df_random_effects_default['Fixed slope trend'] = lmm_default.fe_params.loc['trend']

df_random_effects_default['Fixed slope Is_Public_Holiday'] = lmm_default.fe_params.loc['Is_Public_Holiday']

df_random_effects_default['Fixed slope hdd'] = lmm_default.fe_params.loc['hdd']

In [25]:
# Rename columns for clarity

df_random_effects_default = df_random_effects_default.rename({

    'pce': 'Random intercept',

    'hdd': 'Random slope hdd'

}, axis=1)


In [26]:
# Calculate the combined (fixed + random) effects

df_random_effects_default['INTERCEPT LMM'] = df_random_effects_default['Random intercept'] + df_random_effects_default['Fixed intercept']

df_random_effects_default['SLOPE LMM hdd'] = df_random_effects_default['Random slope hdd'] + df_random_effects_default['Fixed slope hdd']

In [27]:
# delete id needed

df_random_effects = df_random_effects_default


In [28]:
df_random_effects = df_random_effects.reset_index().rename(columns={'index': 'pce'})


In [29]:
df_random_effects

,pce,Random intercept,Random slope hdd,Fixed intercept,Fixed slope trend,Fixed slope Is_Public_Holiday,Fixed slope hdd,INTERCEPT LMM,SLOPE LMM hdd
0,22114761187602,7.051824,-0.434997,8.565324,0.005579,-3.646053,5.823298,15.617148,5.388301
1,22114761188829,27.092570,-1.717639,8.565324,0.005579,-3.646053,5.823298,35.657895,4.105659
2,22114761191548,-9.709267,0.617184,8.565324,0.005579,-3.646053,5.823298,-1.143943,6.440481
3,22114761192886,-1.161910,0.075395,8.565324,0.005579,-3.646053,5.823298,7.403415,5.898692
4,22114761202808,4.526642,-0.287282,8.565324,0.005579,-3.646053,5.823298,13.091966,5.536016
...,...,...,...,...,...,...,...,...,...
7952,22157452873546,-3.840480,0.247940,8.565324,0.005579,-3.646053,5.823298,4.724845,6.071238
7953,22157452876350,-7.247012,0.458581,8.565324,0.005579,-3.646053,5.823298,1.318313,6.281878
7954,22157452887265,9.284888,-0.562030,8.565324,0.005579,-3.646053,5.823298,17.850213,5.261267
7955,22157452890759,-3.651189,0.237431,8.565324,0.005579,-3.646053,5.823298,4.914136,6.060729


In [30]:
merged_df = df_random_effects


In [31]:
data_nona_unique = data_nona[['pce', 'count']].drop_duplicates(subset='pce')


In [33]:
# Convert 'pce' column in both DataFrames to string

merged_df['pce'] = merged_df['pce'].astype(str)

data_nona_unique['pce'] = data_nona_unique['pce'].astype(str)

merged_df=pd.merge(merged_df, data_nona_unique[['pce', 'count']], on='pce', how='left')


In [34]:
merged_df


,pce,Random intercept,Random slope hdd,Fixed intercept,Fixed slope trend,Fixed slope Is_Public_Holiday,Fixed slope hdd,INTERCEPT LMM,SLOPE LMM hdd,count
0,22114761187602,7.051824,-0.434997,8.565324,0.005579,-3.646053,5.823298,15.617148,5.388301,466
1,22114761188829,27.092570,-1.717639,8.565324,0.005579,-3.646053,5.823298,35.657895,4.105659,290
2,22114761191548,-9.709267,0.617184,8.565324,0.005579,-3.646053,5.823298,-1.143943,6.440481,445
3,22114761192886,-1.161910,0.075395,8.565324,0.005579,-3.646053,5.823298,7.403415,5.898692,358
4,22114761202808,4.526642,-0.287282,8.565324,0.005579,-3.646053,5.823298,13.091966,5.536016,470
...,...,...,...,...,...,...,...,...,...,...
7952,22157452873546,-3.840480,0.247940,8.565324,0.005579,-3.646053,5.823298,4.724845,6.071238,266
7953,22157452876350,-7.247012,0.458581,8.565324,0.005579,-3.646053,5.823298,1.318313,6.281878,470
7954,22157452887265,9.284888,-0.562030,8.565324,0.005579,-3.646053,5.823298,17.850213,5.261267,451
7955,22157452890759,-3.651189,0.237431,8.565324,0.005579,-3.646053,5.823298,4.914136,6.060729,225
